#  Get daily runoff data from 3-hourly GLDAS data collection

### Import libraries

To run this example, you will need to import the following libraries, which you should have already installed. 

Note that many (but not all) of these libraries are included in the nasa-gesdisc.yml file, which can be found (along with instructions for installing libraries) in the gesdisc-tutorials Github repository [**here**](https://github.com/nasa/gesdisc-tutorials/tree/main/environments). 

In [1]:
import boto3
import xarray
import numpy as np
import fsspec
import requests
import json
from IPython.display import display, Markdown
import re
from cmr import VariableQuery

### Check AWS region

NASA GES DISC data are stored in the AWS us-west-2 region. If you are computing in a different AWS region, then you will have to move the data outside us-west-2 in order to perform operations on it, and you will lose the computational efficiency that comes with working "next to the data" in the cloud.

In [2]:
if (boto3.client('s3').meta.region_name == 'us-west-2'):
    display(Markdown('### us-west-2 Region Check: &#x2705;'))
else:
    display(Markdown('### us-west-2 Region Check: &#10060;'))
    raise ValueError('Your notebook is not running inside the AWS us-west-2 region, and will not be able to directly access NASA Earthdata S3 buckets')

### us-west-2 Region Check: &#x2705;

### Read GLDAS data from the Giovanni zarr store

Currently, GLDAS_NOAH025_3H is the only GES DISC collection available in zarr format. Consult the GLDAS readme to find variable shortnames.

In [3]:
# Get list of all variables available as zarr stores
api = VariableQuery()
all_vars = api.get_all()
zarr_stores = []
for variable_entry in all_vars:
    try:
        if variable_entry['instance_information']['format'] == 'zarr':
            zarr_stores.append(variable_entry)
    except KeyError:
        continue

# Search the list of all zarr variables for the variable of interest
short_name = "GLDAS_NOAH025_3H" # collection shortname
version = "2.0" # collection version
variable = "/Qs_acc" # runoff (consult readme file to find list of shortnames)
pattern = short_name + '.*' + version + '.*' + variable + '.*'

zarrs = []
for store in zarr_stores:
        try:
            if re.match(pattern, store["native_id"]):
                zarrs.append(store)
        except KeyError:
            continue

# Extract metadata used to access the zarr file
instance_information = zarrs[0]['instance_information']

# Retrieve credentials endpoint (specific to Giovanni Zarr stores only)
credentials_endpoint = instance_information['direct_distribution_information']['s_3_credentials_api_endpoint']
credentials = json.loads(requests.get(credentials_endpoint).text)

# Open zarr store with xarray
s3 = fsspec.filesystem(
    "s3",
    key=credentials["AccessKeyId"],
    secret=credentials["SecretAccessKey"],
    token=credentials["SessionToken"],
)
store = s3.get_mapper(instance_information['url'])
ds_surface_runoff = xarray.open_zarr(store) # this is a dask dataset

### Resample from 3-hourly to daily resolution

A couple important notes for resampling:
* The GLDAS data here have extra fillvalues added onto the end of the xarray DataArray as padding. These must be removed (e.g. by subsetting the data to a particular time range), or resample() will fail.
* Calling resample() loads the GLDAS data into memory. If you are processing long time periods, this can fill up your RAM. It may be best to process in shorter increments (e.g. one year at a time).

In [4]:
# Subset data to a particular time range (e.g. one year)
time_range = (np.datetime64('2014-01-01'),np.datetime64('2014-12-31')) # desired time range
start_time, end_time = time_range
time_condition = np.logical_and(ds_surface_runoff.time >= start_time, ds_surface_runoff.time <=end_time)
subset = ds_surface_runoff.sel(time=time_condition)
daily_total_runoff = subset['variable'].resample(time='1D').sum()

### Save runoff data to a NetCDF file

In [5]:
# Save the daily data for this year
del daily_total_runoff.time.attrs['_FillValue'] # get rid of bad fillvalue for time attribute that prevents write
output_file = f"./Data/potomac_daily_runoff_2014_directS3_gldas_zarr.nc"
daily_total_runoff.to_netcdf(output_file)